# Combined Denoising + Regression Model: Evaluation

This notebook evaluates a trained `LitS4CombinedModel` (the `S4DCombinedModel` architecture)
which jointly performs:
1. **Denoising** – `S4DSeq2SeqModel` maps a noisy I/Q sequence to a denoised sequence
2. **Regression** – `S4DModel` reads the denoised sequence and predicts energy + pitch angle

Batches produced by `LitCombinedDataModule` have the form `(x_noisy, x_clean, y, obs)`.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import torch
import numpy as np
import matplotlib.pyplot as plt
import yaml
from scipy import signal as sp_signal

from src.models.model import LitS4CombinedModel
from src.models.networks import S4DCombinedModel
from src.data.data import LitCombinedDataModule

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

## 1. Configuration

Set the paths to your checkpoint and config file, and the directory containing your HDF5 data.

In [ ]:
# ── Update these paths ────────────────────────────────────────────────────────
CHECKPOINT_PATH = '../runs/combined/lightning_logs/<run_id>/checkpoints/last.ckpt'
CONFIG_PATH     = '../runs/combined/config.yaml'  # saved alongside the checkpoint
DATA_DIR        = '/path/to/data/'                # directory with HDF5 files
# ─────────────────────────────────────────────────────────────────────────────

## 2. Load Config and Data Module

In [ ]:
with open(CONFIG_PATH) as f:
    config = yaml.safe_load(f)

data_cfg = config['data']['init_args']

dm = LitCombinedDataModule(
    inputs      = data_cfg['inputs'],
    variables   = data_cfg['variables'],
    observables = data_cfg['observables'],
    data_dir    = DATA_DIR,
    cutoff      = data_cfg.get('cutoff', 8192),
    noise_const = data_cfg.get('noise_const', 1.0),
    apply_filter= data_cfg.get('apply_filter', False),
    batch_size  = 32,   # smaller batch for visualisation
    num_workers = 0,
)

test_loader = dm.test_dataloader()
print(f'Test batches  : {len(test_loader)}')
print(f'Input channels: {data_cfg["inputs"]}')
print(f'Targets       : {data_cfg["variables"]}')
print(f'Observables   : {data_cfg["observables"]}')

## 3. Load Model

In [ ]:
model_cfg = config['model']['init_args']
enc_cfg   = model_cfg['encoder']['init_args']

encoder = S4DCombinedModel(
    d_input            = enc_cfg.get('d_input', 2),
    d_output           = enc_cfg.get('d_output', 2),
    denoiser_d_model   = enc_cfg.get('denoiser_d_model', 128),
    denoiser_n_layers  = enc_cfg.get('denoiser_n_layers', 6),
    denoiser_dropout   = enc_cfg.get('denoiser_dropout', 0.0),
    denoiser_prenorm   = enc_cfg.get('denoiser_prenorm', False),
    regressor_d_model  = enc_cfg.get('regressor_d_model', 128),
    regressor_n_layers = enc_cfg.get('regressor_n_layers', 6),
    regressor_dropout  = enc_cfg.get('regressor_dropout', 0.0),
    regressor_prenorm  = enc_cfg.get('regressor_prenorm', False),
    regressor_fc_hidden= enc_cfg.get('regressor_fc_hidden', [64, 32]),
)

model = LitS4CombinedModel.load_from_checkpoint(
    CHECKPOINT_PATH,
    encoder=encoder,
)
model = model.to(device).eval()
print('Model loaded successfully')
print(f'  Denoiser : d_model={enc_cfg.get("denoiser_d_model", 128)}, '
      f'n_layers={enc_cfg.get("denoiser_n_layers", 6)}')
print(f'  Regressor: d_model={enc_cfg.get("regressor_d_model", 128)}, '
      f'n_layers={enc_cfg.get("regressor_n_layers", 6)}')

## 4. Visualise Example Inputs

Plot a few noisy / clean pairs straight from the dataloader.

In [ ]:
N_EXAMPLES = 4
CHANNEL_LABELS = ['I channel', 'Q channel']

x_noisy_batch, x_clean_batch, y_batch, obs_batch = next(iter(test_loader))
cutoff = x_noisy_batch.shape[1]
t = np.arange(cutoff)

fig, axes = plt.subplots(N_EXAMPLES, 2, figsize=(14, 2.5 * N_EXAMPLES), sharex=True)

for i in range(N_EXAMPLES):
    for ch, label in enumerate(CHANNEL_LABELS):
        ax = axes[i, ch]
        ax.plot(t, x_noisy_batch[i, :, ch].numpy(), color='tab:blue',   alpha=0.6, lw=0.6, label='Noisy')
        ax.plot(t, x_clean_batch[i, :, ch].numpy(), color='tab:orange', lw=1.2,            label='Clean')
        ax.set_ylabel('Amplitude (norm.)')
        ax.set_title(f'Sample {i+1} — {label}')
        if i == 0:
            ax.legend(loc='upper right', fontsize=8)

axes[-1, 0].set_xlabel('Time sample')
axes[-1, 1].set_xlabel('Time sample')
fig.suptitle('Example inputs: noisy vs clean I/Q signals', fontsize=13)
fig.tight_layout()
plt.show()

## 5. Run the Model and Visualise Denoising Predictions

In [ ]:
with torch.no_grad():
    x_denoised_batch, y_pred_batch = model(x_noisy_batch.to(device))
    x_denoised_batch = x_denoised_batch.cpu()
    y_pred_batch     = y_pred_batch.cpu()

fig, axes = plt.subplots(N_EXAMPLES, 2, figsize=(14, 2.5 * N_EXAMPLES), sharex=True)

for i in range(N_EXAMPLES):
    for ch, label in enumerate(CHANNEL_LABELS):
        ax = axes[i, ch]
        ax.plot(t, x_noisy_batch[i, :, ch].numpy(),    color='tab:blue',   alpha=0.4, lw=0.6, label='Noisy input')
        ax.plot(t, x_clean_batch[i, :, ch].numpy(),    color='tab:orange', lw=1.2,            label='Clean target')
        ax.plot(t, x_denoised_batch[i, :, ch].numpy(), color='tab:green',  lw=1.2, ls='--',   label='Denoised')
        ax.set_ylabel('Amplitude (norm.)')
        ax.set_title(f'Sample {i+1} — {label}')
        if i == 0:
            ax.legend(loc='upper right', fontsize=8)

axes[-1, 0].set_xlabel('Time sample')
axes[-1, 1].set_xlabel('Time sample')
fig.suptitle('Denoising predictions: noisy input / clean target / model output', fontsize=13)
fig.tight_layout()
plt.show()

## 6. Zoom In on a Signal Region

In [ ]:
WINDOW_START = 0
WINDOW_LEN   = 512
SAMPLE_IDX   = 0

sl     = slice(WINDOW_START, WINDOW_START + WINDOW_LEN)
t_zoom = np.arange(WINDOW_START, WINDOW_START + WINDOW_LEN)

fig, axes = plt.subplots(1, 2, figsize=(14, 3.5))

for ch, label in enumerate(CHANNEL_LABELS):
    ax = axes[ch]
    ax.plot(t_zoom, x_noisy_batch[SAMPLE_IDX, sl, ch].numpy(),    color='tab:blue',   alpha=0.5, lw=0.8, label='Noisy input')
    ax.plot(t_zoom, x_clean_batch[SAMPLE_IDX, sl, ch].numpy(),    color='tab:orange', lw=1.5,            label='Clean target')
    ax.plot(t_zoom, x_denoised_batch[SAMPLE_IDX, sl, ch].numpy(), color='tab:green',  lw=1.5, ls='--',   label='Denoised')
    ax.set_xlabel('Time sample')
    ax.set_ylabel('Amplitude (norm.)')
    ax.set_title(f'Sample {SAMPLE_IDX+1} — {label} (zoom)')
    ax.legend(fontsize=8)

fig.tight_layout()
plt.show()

## 7. Power Spectral Density Comparison

Compare the PSD of the noisy input, clean target, and denoised prediction.

In [ ]:
FS = 403e6  # sampling frequency in Hz

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ch, label in enumerate(CHANNEL_LABELS):
    ax = axes[ch]
    for sig, name, color in [
        (x_noisy_batch[SAMPLE_IDX, :, ch].numpy(),    'Noisy input',  'tab:blue'),
        (x_clean_batch[SAMPLE_IDX, :, ch].numpy(),    'Clean target', 'tab:orange'),
        (x_denoised_batch[SAMPLE_IDX, :, ch].numpy(), 'Denoised',     'tab:green'),
    ]:
        f, pxx = sp_signal.periodogram(sig, fs=FS)
        ax.semilogy(f / 1e6, pxx, lw=0.8, label=name, color=color)
    ax.set_xlabel('Frequency [MHz]')
    ax.set_ylabel('PSD [arb/Hz]')
    ax.set_title(f'{label} — PSD')
    ax.legend(fontsize=8)

fig.suptitle('Power Spectral Density: noisy / clean / denoised', fontsize=13)
fig.tight_layout()
plt.show()

## 8. Run Full Test Set — Collect Predictions

Iterate over the full test dataloader to collect denoising MSE and regression predictions.

In [ ]:
import tqdm

true_list    = []
pred_list    = []
meta_list    = []
mse_list     = []
max_powers   = []
summed_powers= []

for x_noisy, x_clean, y, obs in tqdm.tqdm(test_loader, desc='Evaluating'):
    with torch.no_grad():
        x_denoised, y_pred = model(x_noisy.to(device))
        x_denoised = x_denoised.cpu()
        y_pred     = y_pred.cpu()

    true_list.append(y.numpy())
    pred_list.append(y_pred.numpy())
    meta_list.append(obs.numpy())

    # Per-sample denoising MSE (mean over L and C)
    mse = ((x_denoised - x_clean) ** 2).mean(dim=(1, 2))
    mse_list.append(mse.numpy())

    # PSD-based power features from the noisy time series
    for i in range(x_noisy.shape[0]):
        f_i, pxx_i = sp_signal.periodogram(x_noisy[i, :, 0].numpy(), fs=FS)
        f_q, pxx_q = sp_signal.periodogram(x_noisy[i, :, 1].numpy(), fs=FS)
        pxx_sum = pxx_i + pxx_q
        noise_floor = np.sum(pxx_sum[:400]) + 1e-30
        normed = pxx_sum / noise_floor
        max_powers.append(normed.max())
        summed_powers.append(normed.sum())

true_all        = np.concatenate(true_list,  axis=0)
pred_all        = np.concatenate(pred_list,  axis=0)
meta_all        = np.concatenate(meta_list,  axis=0)
mse_all         = np.concatenate(mse_list)
max_powers      = np.array(max_powers)
summed_powers   = np.array(summed_powers)

print(f'Test samples: {len(true_all)}')

## 9. Undo Normalisation

In [ ]:
# dm.mu and dm.stds are set during dataset construction via Welford stats
pred_post = pred_all * dm.stds + dm.mu
true_post = true_all * dm.stds + dm.mu

true_energy  = true_post[:, 0]
pred_energy  = pred_post[:, 0]
energy_error = true_energy - pred_energy

print(f'Energy error  mean : {energy_error.mean():.3f} eV')
print(f'Energy error  std  : {energy_error.std():.3f} eV')
print(f'Energy error |<3eV|: {(np.abs(energy_error) < 3).mean()*100:.1f} %')

## 10. Per-Sample Denoising MSE Distribution

In [ ]:
print(f'Test MSE mean  : {mse_all.mean():.6f}')
print(f'Test MSE median: {np.median(mse_all):.6f}')
print(f'Test MSE std   : {mse_all.std():.6f}')

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(mse_all, bins=80, color='steelblue', edgecolor='white', linewidth=0.4)
ax.axvline(mse_all.mean(),      color='tab:red',    lw=1.5, ls='--', label=f'Mean = {mse_all.mean():.4f}')
ax.axvline(np.median(mse_all),  color='tab:orange', lw=1.5, ls=':',  label=f'Median = {np.median(mse_all):.4f}')
ax.set_xlabel('Per-sample denoising MSE (normalised amplitude)')
ax.set_ylabel('Count')
ax.set_title('Test-set denoising MSE distribution')
ax.legend()
fig.tight_layout()
plt.show()

## 11. Regression: Energy Error Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(energy_error, bins=100, color='steelblue', edgecolor='white', linewidth=0.4)
ax.axvline( 3, color='r', lw=1.5, ls='--', alpha=0.7, label='±3 eV')
ax.axvline(-3, color='r', lw=1.5, ls='--', alpha=0.7)
ax.axvline(energy_error.mean(), color='tab:orange', lw=1.5, label=f'Mean = {energy_error.mean():.2f} eV')
ax.set_xlabel('Energy error (true − predicted) [eV]')
ax.set_ylabel('Count')
ax.set_title('Energy reconstruction error distribution')
ax.set_xlim(-50, 50)
ax.legend()
fig.tight_layout()
plt.show()

## 12. Energy Error vs Axial / Carrier Frequency

In [ ]:
avg_axial_freq   = meta_all[:, 0]
avg_carrier_freq = meta_all[:, 1]

fig, ax = plt.subplots(figsize=(8, 5))
sc = ax.scatter(
    avg_axial_freq, avg_carrier_freq,
    c=np.abs(energy_error), s=1, cmap='viridis', vmax=20,
)
cbar = plt.colorbar(sc, ax=ax)
cbar.set_label(r'$|\mathrm{Energy}_{\mathrm{true}} - \mathrm{Energy}_{\mathrm{pred}}|$ [eV]')
ax.set_xlabel('Average Axial Frequency [Hz]')
ax.set_ylabel('Average Carrier Frequency [Hz]')
ax.set_title('Energy error vs signal frequencies')
fig.tight_layout()
plt.show()

## 13. Energy Error vs Summed / Max Power

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, powers, xlabel in [
    (axes[0], summed_powers, 'Summed power [arb]'),
    (axes[1], max_powers,    'Max power [arb]'),
]:
    h = ax.hist2d(powers, energy_error, bins=(100, 250), cmin=1)
    ax.axhline( 3, color='r', alpha=0.6, lw=1.2, label='±3 eV')
    ax.axhline(-3, color='r', alpha=0.6, lw=1.2)
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Energy error [eV]')
    ax.set_ylim(-20, 20)
    ax.legend(fontsize=8)
    plt.colorbar(h[3], ax=ax)

fig.suptitle('Energy error vs signal power (zoomed to ±20 eV)', fontsize=13)
fig.tight_layout()
plt.show()

## 14. Pitch Angle and Radius Distributions

Compare full test-set distributions with the subset of well-reconstructed events (|ΔE| < 3 eV).

In [ ]:
recon_mask = np.abs(energy_error) < 3

true_pitch = true_post[:, 1]
true_radii = meta_all[:, 2]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(true_pitch,              bins=np.linspace(88, 90, 100), density=True, alpha=0.5, label='All events')
axes[0].hist(true_pitch[recon_mask],  bins=np.linspace(88, 90, 100), density=True, alpha=0.5, label='|ΔE| < 3 eV')
axes[0].set_xlabel(r'Pitch Angle [$^\circ$]')
axes[0].set_ylabel('Probability Density')
axes[0].legend()

axes[1].hist(true_radii,             bins=np.linspace(0, 0.007, 100), density=True, alpha=0.5, label='All events')
axes[1].hist(true_radii[recon_mask], bins=np.linspace(0, 0.007, 100), density=True, alpha=0.5, label='|ΔE| < 3 eV')
axes[1].set_xlabel('Radius [m]')
axes[1].set_ylabel('Probability Density')
axes[1].legend()

fig.suptitle('Pitch angle and radius distributions', fontsize=13)
fig.tight_layout()
plt.show()

## 15. Denoising MSE vs Regression Energy Error

Does better denoising lead to better regression? Scatter the per-sample denoising MSE against the energy error.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
h = ax.hist2d(mse_all, np.abs(energy_error), bins=(80, 80), cmin=1, cmap='viridis')
ax.set_xlabel('Denoising MSE (per sample)')
ax.set_ylabel('|Energy error| [eV]')
ax.set_title('Denoising quality vs regression accuracy')
ax.set_ylim(0, 50)
plt.colorbar(h[3], ax=ax)
fig.tight_layout()
plt.show()